# Toxicity Detection Model — Training Pipeline

**Bidirectional LSTM** classifier for detecting toxic messages in chat.

**What this notebook does:**
1. Downloads the Jigsaw Toxic Comment dataset (or loads a local CSV)
2. Preprocesses and cleans text data
3. Builds a vocabulary (matches `ai/toxicity/utils/vocabulary.py`)
4. Defines the BiLSTM classifier (matches `ai/toxicity/model/lstm_classifier.py`)
5. Trains the model with class-weighted loss
6. Evaluates with accuracy, precision, recall, F1 & confusion matrix
7. Exports `model.pt` and `vocab.json` for backend inference

**Runtime:** Use GPU if available (Kaggle / Colab). Falls back to CPU automatically.

In [ ]:
# ============================================================
# Cell 1 — Install dependencies (run once)
# ============================================================
!pip install -q torch numpy matplotlib scikit-learn pandas

In [ ]:
# ============================================================
# Cell 2 — Imports & Device Setup
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import json
import re
import os
import random
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Auto-detect GPU / CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Dataset — Jigsaw Toxic Comment Classification

We use a CSV with columns `comment_text` and toxicity labels.  
If you have the Jigsaw dataset from Kaggle, place `train.csv` in the same directory.  
Otherwise, this notebook generates a sample dataset for demonstration.

In [ ]:
# ============================================================
# Cell 3 — Load dataset
# ============================================================
# Option A: Load Jigsaw Toxic Comment dataset (from Kaggle)
# Download from: https://www.kaggle.com/c/jigsaw-toxic-comment-classification-challenge/data
# Place train.csv in the working directory.

CSV_PATH = "train.csv"  # Change this path if needed

if os.path.exists(CSV_PATH):
    print(f"Loading dataset from {CSV_PATH}...")
    df = pd.read_csv(CSV_PATH)
    # Jigsaw has multiple toxicity columns — combine into a single binary label
    toxicity_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
    available_cols = [c for c in toxicity_cols if c in df.columns]
    if available_cols:
        df["is_toxic"] = (df[available_cols].sum(axis=1) > 0).astype(int)
    elif "is_toxic" not in df.columns and "label" in df.columns:
        df["is_toxic"] = df["label"]
    print(f"Loaded {len(df)} samples")
else:
    # Option B: Generate a sample dataset for demonstration
    print("Jigsaw dataset not found. Generating sample dataset...")
    non_toxic_samples = [
        "hello how are you doing today",
        "great job on the presentation",
        "can we meet tomorrow at noon",
        "thanks for your help with the project",
        "the weather is really nice today",
        "i enjoyed the movie last night",
        "could you please send me the report",
        "happy birthday hope you have a great day",
        "looking forward to the weekend",
        "that was a really interesting discussion",
        "i appreciate your feedback",
        "lets grab lunch sometime this week",
        "congratulations on your promotion",
        "have a wonderful evening",
        "the team did an excellent job",
        "i really liked your idea in the meeting",
        "see you at the conference next week",
        "thanks for sharing this article",
        "what a beautiful sunset",
        "hope your day is going well",
    ]
    toxic_samples = [
        "you are so stupid and worthless",
        "shut up nobody cares about you",
        "i hate everything about you",
        "you are the worst person ever",
        "go away nobody wants you here",
        "you are an idiot and a fool",
        "this is the dumbest thing ever",
        "you disgust me completely",
        "everyone hates you just leave",
        "you are totally useless and pathetic",
        "what a terrible and awful person",
        "you make me sick with your nonsense",
        "nobody likes you at all",
        "you are a complete waste of time",
        "get lost you annoying fool",
        "you are so ugly and repulsive",
        "your opinion is garbage and meaningless",
        "die in a fire you moron",
        "you deserve nothing but misery",
        "shut your mouth you imbecile",
    ]

    # Augment by slight variations
    def augment_text(texts, multiplier=50):
        augmented = []
        for text in texts:
            augmented.append(text)
            words = text.split()
            for _ in range(multiplier - 1):
                new_words = words.copy()
                # Randomly drop/duplicate/swap words
                op = random.choice(["drop", "dup", "swap", "none"])
                if op == "drop" and len(new_words) > 2:
                    idx = random.randint(0, len(new_words) - 1)
                    new_words.pop(idx)
                elif op == "dup" and len(new_words) > 1:
                    idx = random.randint(0, len(new_words) - 1)
                    new_words.insert(idx, new_words[idx])
                elif op == "swap" and len(new_words) > 2:
                    i, j = random.sample(range(len(new_words)), 2)
                    new_words[i], new_words[j] = new_words[j], new_words[i]
                augmented.append(" ".join(new_words))
        return augmented

    non_toxic_aug = augment_text(non_toxic_samples, multiplier=50)
    toxic_aug = augment_text(toxic_samples, multiplier=50)

    texts = non_toxic_aug + toxic_aug
    labels = [0] * len(non_toxic_aug) + [1] * len(toxic_aug)

    df = pd.DataFrame({"comment_text": texts, "is_toxic": labels})
    df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    print(f"Generated {len(df)} samples")

print(f"\nLabel distribution:")
print(df["is_toxic"].value_counts())
print(f"\nSample data:")
df.head(10)

In [ ]:
# ============================================================
# Cell 4 — Text preprocessing
# ============================================================
def clean_text(text):
    """Matches ai/common/preprocessing/text_utils.py"""
    text = str(text).lower().strip()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"[^a-zA-Z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["clean_text"] = df["comment_text"].apply(clean_text)

# Remove empty rows
df = df[df["clean_text"].str.len() > 0].reset_index(drop=True)
print(f"Samples after cleaning: {len(df)}")
print(f"\nExamples:")
for _, row in df.head(5).iterrows():
    label = "TOXIC" if row["is_toxic"] else "clean"
    print(f"  [{label:5s}] {row['clean_text'][:80]}")

## 2. Vocabulary

Matches `ai/toxicity/utils/vocabulary.py` exactly so exported `vocab.json` works with the backend.

In [ ]:
# ============================================================
# Cell 5 — Vocabulary class (matches ai/toxicity/utils/vocabulary.py)
# ============================================================
class ToxicityVocabulary:
    PAD_TOKEN = "<PAD>"
    UNK_TOKEN = "<UNK>"

    def __init__(self, max_vocab_size=20000):
        self.word2idx = {self.PAD_TOKEN: 0, self.UNK_TOKEN: 1}
        self.idx2word = {0: self.PAD_TOKEN, 1: self.UNK_TOKEN}
        self.word_count = Counter()
        self.max_vocab_size = max_vocab_size

    def build_from_texts(self, texts):
        for text in texts:
            tokens = self.tokenize(text)
            self.word_count.update(tokens)
        most_common = self.word_count.most_common(self.max_vocab_size - 2)
        for word, _ in most_common:
            idx = len(self.word2idx)
            self.word2idx[word] = idx
            self.idx2word[idx] = word

    @staticmethod
    def tokenize(text):
        text = text.lower().strip()
        text = re.sub(r"[^a-zA-Z0-9\s']", " ", text)
        return text.split()

    def encode(self, text, max_len=200):
        tokens = self.tokenize(text)
        ids = [self.word2idx.get(t, self.word2idx[self.UNK_TOKEN]) for t in tokens[:max_len]]
        ids += [self.word2idx[self.PAD_TOKEN]] * (max_len - len(ids))
        return ids

    def save(self, path):
        data = {"word2idx": self.word2idx, "idx2word": {str(k): v for k, v in self.idx2word.items()}}
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w") as f:
            json.dump(data, f)

    def load(self, path):
        with open(path, "r") as f:
            data = json.load(f)
        self.word2idx = data["word2idx"]
        self.idx2word = {int(k): v for k, v in data["idx2word"].items()}

    def __len__(self):
        return len(self.word2idx)

In [ ]:
# ============================================================
# Cell 6 — Build vocabulary
# ============================================================
MAX_VOCAB = 20000
MAX_LEN = 200  # max tokens per message

vocab = ToxicityVocabulary(max_vocab_size=MAX_VOCAB)
vocab.build_from_texts(df["clean_text"].tolist())
print(f"Vocabulary size: {len(vocab)}")
print(f"Max sequence length: {MAX_LEN}")

# Show most common words
print(f"\nTop 20 words:")
for word, count in vocab.word_count.most_common(20):
    print(f"  {word}: {count}")

## 3. Dataset & DataLoader

In [ ]:
# ============================================================
# Cell 7 — PyTorch Dataset & DataLoader
# ============================================================
class ToxicityDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        input_ids = self.vocab.encode(self.texts[idx], self.max_len)
        label = self.labels[idx]
        return torch.tensor(input_ids, dtype=torch.long), torch.tensor(label, dtype=torch.long)

# Train / Val / Test split (80/10/10)
n = len(df)
indices = np.random.permutation(n)
train_end = int(0.8 * n)
val_end = int(0.9 * n)

train_df = df.iloc[indices[:train_end]].reset_index(drop=True)
val_df = df.iloc[indices[train_end:val_end]].reset_index(drop=True)
test_df = df.iloc[indices[val_end:]].reset_index(drop=True)

BATCH_SIZE = 64

train_dataset = ToxicityDataset(train_df["clean_text"].tolist(), train_df["is_toxic"].tolist(), vocab, MAX_LEN)
val_dataset = ToxicityDataset(val_df["clean_text"].tolist(), val_df["is_toxic"].tolist(), vocab, MAX_LEN)
test_dataset = ToxicityDataset(test_df["clean_text"].tolist(), test_df["is_toxic"].tolist(), vocab, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print(f"Batches per epoch: {len(train_loader)}")
print(f"\nTrain label distribution:")
print(f"  Non-toxic: {sum(1 for l in train_df['is_toxic'] if l == 0)}")
print(f"  Toxic:     {sum(1 for l in train_df['is_toxic'] if l == 1)}")

## 4. Model Architecture

**Bidirectional LSTM Classifier** — matches `ai/toxicity/model/lstm_classifier.py` exactly.

In [ ]:
# ============================================================
# Cell 8 — Model (mirrors ai/toxicity/model/lstm_classifier.py)
# ============================================================
class ToxicityLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=128, num_layers=2, dropout=0.3, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)  # *2 for bidirectional

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        lstm_out, (hidden, _) = self.lstm(embedded)

        # Concatenate final forward and backward hidden states
        hidden_fwd = hidden[-2]
        hidden_bwd = hidden[-1]
        hidden_cat = torch.cat((hidden_fwd, hidden_bwd), dim=1)

        out = self.dropout(hidden_cat)
        out = self.fc(out)
        return out


# Hyperparameters
EMBED_DIM = 100
HIDDEN_DIM = 128
NUM_LAYERS = 2
DROPOUT = 0.3
NUM_CLASSES = 2
LEARNING_RATE = 1e-3
NUM_EPOCHS = 15

model = ToxicityLSTM(len(vocab), EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT, NUM_CLASSES).to(device)

# Class-weighted loss to handle imbalanced data
toxic_count = train_df["is_toxic"].sum()
non_toxic_count = len(train_df) - toxic_count
weight = torch.tensor([1.0, non_toxic_count / max(toxic_count, 1)], dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=weight)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(f"Class weights: {weight.tolist()}")
print(model)

## 5. Training Loop

In [ ]:
# ============================================================
# Cell 9 — Training loop
# ============================================================
train_losses = []
val_losses = []
train_accs = []
val_accs = []
best_val_loss = float("inf")

for epoch in range(1, NUM_EPOCHS + 1):
    # ---- Train ----
    model.train()
    epoch_loss = 0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_train_loss = epoch_loss / len(train_loader)
    train_acc = correct / total
    train_losses.append(avg_train_loss)
    train_accs.append(train_acc)

    # ---- Validate ----
    model.eval()
    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_val_loss = val_loss / len(val_loader)
    val_acc = correct / total
    val_losses.append(avg_val_loss)
    val_accs.append(val_acc)
    scheduler.step(avg_val_loss)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "best_toxicity_model.pt")

    lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | Train Loss={avg_train_loss:.4f} Acc={train_acc:.4f} | Val Loss={avg_val_loss:.4f} Acc={val_acc:.4f} | LR={lr:.6f}")

print(f"\nBest val loss: {best_val_loss:.4f}")

In [ ]:
# ============================================================
# Cell 10 — Plot training curves
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses, label="Train Loss")
ax1.plot(val_losses, label="Val Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Toxicity Model — Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(train_accs, label="Train Acc")
ax2.plot(val_accs, label="Val Acc")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Toxicity Model — Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Evaluation — Test Set Metrics

In [ ]:
# ============================================================
# Cell 11 — Evaluate on test set
# ============================================================
model.load_state_dict(torch.load("best_toxicity_model.pt", map_location=device))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

# Metrics
acc = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, zero_division=0)
rec = recall_score(all_labels, all_preds, zero_division=0)
f1 = f1_score(all_labels, all_preds, zero_division=0)

print("=" * 50)
print("Test Set Evaluation")
print("=" * 50)
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=["non-toxic", "toxic"]))

In [ ]:
# ============================================================
# Cell 12 — Confusion Matrix
# ============================================================
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["non-toxic", "toxic"])
ax.set_yticklabels(["non-toxic", "toxic"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix")

for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i][j]), ha="center", va="center",
                color="white" if cm[i][j] > cm.max() / 2 else "black", fontsize=16)

plt.colorbar(im)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 13 — Test with sample messages
# ============================================================
def predict_toxicity(text):
    """Predict toxicity of a text string."""
    cleaned = clean_text(text)
    input_ids = torch.tensor([vocab.encode(cleaned, MAX_LEN)]).to(device)
    with torch.no_grad():
        logits = model(input_ids)
        probs = torch.softmax(logits, dim=-1)
        pred_class = probs.argmax(dim=-1).item()
        confidence = probs[0][pred_class].item()
    label = "TOXIC" if pred_class == 1 else "non-toxic"
    return label, confidence

test_messages = [
    "hey, how are you doing today?",
    "you are such an idiot",
    "thanks for helping me out!",
    "shut up you fool nobody asked you",
    "great job on the project",
    "i hate you so much",
    "can we reschedule the meeting?",
    "you are completely worthless",
    "happy birthday!",
    "go away and never come back",
]

print("=" * 60)
print("Toxicity Detection — Sample Predictions")
print("=" * 60)
for msg in test_messages:
    label, conf = predict_toxicity(msg)
    print(f"  [{label:9s} {conf:.2%}] {msg}")

## 7. Export Model for Backend

Saves `model.pt` and `vocab.json` that the FastAPI backend can load from `ai/saved_models/toxicity/`.

In [ ]:
# ============================================================
# Cell 14 — Save model & vocab for backend inference
# ============================================================
import shutil

EXPORT_DIR = "toxicity_export"
BACKEND_DIR = os.path.join("..", "ai", "saved_models", "toxicity")
os.makedirs(EXPORT_DIR, exist_ok=True)

# Save model weights (CPU for portability)
model.cpu()
torch.save(model.state_dict(), os.path.join(EXPORT_DIR, "model.pt"))

# Save vocabulary
vocab.save(os.path.join(EXPORT_DIR, "vocab.json"))

print(f"Exported to '{EXPORT_DIR}/'")
print(f"  model.pt   — {os.path.getsize(os.path.join(EXPORT_DIR, 'model.pt')) / 1e6:.1f} MB")
print(f"  vocab.json — {os.path.getsize(os.path.join(EXPORT_DIR, 'vocab.json')) / 1e6:.1f} MB")

# Auto-copy to backend model directory
os.makedirs(BACKEND_DIR, exist_ok=True)
shutil.copy2(os.path.join(EXPORT_DIR, "model.pt"), os.path.join(BACKEND_DIR, "model.pt"))
shutil.copy2(os.path.join(EXPORT_DIR, "vocab.json"), os.path.join(BACKEND_DIR, "vocab.json"))
print(f"\nAuto-copied to '{BACKEND_DIR}/' for backend inference.")

In [ ]:
# ============================================================
# Cell 15 — (Optional) Download from Colab / Kaggle
# ============================================================
# Uncomment the appropriate section:

# --- Google Colab ---
# from google.colab import files
# files.download(os.path.join(EXPORT_DIR, "model.pt"))
# files.download(os.path.join(EXPORT_DIR, "vocab.json"))

# --- Kaggle ---
# import shutil
# shutil.copytree(EXPORT_DIR, "/kaggle/working/toxicity_export")
# print("Files saved to /kaggle/working/toxicity_export/")

print("Uncomment the section above that matches your platform.")